<head>
    <div style="text-align:center">
        <span style="font-size:26px">Overview of the Project</span>
    </div>
</head>
<body>
    <h3>1. The Setup: Bridging the Macro to the Micro</h3>
    <p><span style="font-weight:bold; text-decoration: underline">Objective:</span> <span>"In my first project, I built a macro-level dashboard to understand broader economic spending trends across India.<br><i> However</i>, a Fintech company cannot run targeted marketing campaigns on a whole country. For this second project, I zoomed in to the micro-level to analyze individual customer behavior."</span></p>
    <h3>2. The Methodology: RFM &amp; AI</h3>
    <p><span style="font-weight:bold; text-decoration: underline">Next Step:</span> <span style="font-style: italic">"I started with a database of 2,000 unique users and their transaction histories. To make this data actionable, I used a two-step pipeline:"</span></p>
    <ul>
        <li><span style="font-weight:bold">Step 1: RFM Analysis:</span> <span  class="pro-text">"First, I used Pandas to engineer three core behavioral metrics for every single user: <span style="font-weight:bold">Recency</span> (how many days since their last transaction), <span style="font-weight:bold">Frequency</span> (how often they transact), and <span style="font-weight:bold">Monetary value</span> (their total spend)."</span></li>
        <li><span style="font-weight:bold">Step 2: Unsupervised Learning:</span> <span class="what-to-say">"Next, I scaled that data and fed it into a <span style="font-weight:bold">K-Means Clustering</span> algorithm. Because this is unsupervised machine learning, I didn't give the AI predefined categories. I let the mathematical distances between the users dictate the natural groupings."</span></li>
    </ul>
    <h3>3. The Business Value (The "So What?")</h3>
    <p><span style="font-weight:bold; text-decoration: underline">Final Step:</span> <span style="font-style: italic">"The algorithm successfully grouped my user base into distinct financial personas. This is the exact data a Fintech Growth team uses to generate revenue. For example:"</span></p>
    <ul>
        <li><span style="font-weight:bold">The Whales (High Frequency, High Monetary):</span> <span class="what-to-say">"We target these users with premium Co-Branded Credit Card offers."</span></li>
        <li><span style="font-weight:bold">The Churn Risks (High Recency, Low Frequency):</span> <span class="what-to-say">"These are users who haven't opened the app in months. We target them with aggressive 'Welcome Back' cashback push notifications."</span></li>
    </ul>
    <p>-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------</p>
</body>

<h3>Phase 1: Synthesizing the Customer Base</h3>
<span style="font-size:18px">Our current dataset has transactions, but no users. We need to simulate a realistic user base where some people are "power users" and others are "casual users."
</span>

In [2]:
import pandas as pd
import numpy as np

# 1. Load the pristine dataset from Phase 1
df = pd.read_csv('engineered_upi_master.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])

# 2. Generating 2,000 unique User IDs
num_users = 2000
user_ids = [f'User_{str(i).zfill(4)}' for i in range(1, num_users + 1)] 

'''I needed standard, clean identifiers for my database. Instead of just numbers, I used string formatting with .zfill(4) to generate padded IDs like 'User_0001'. 
This mimics how production SQL databases format primary keys, preventing sorting bugs later."'''

# 3. Assign users to transactions with natural probability
np.random.seed(42)
df['User_ID'] = np.random.choice(user_ids, size=len(df))

''''To make the user assignment more realistic, I used np.random.choice to randomly assign one of the 2,000 User IDs to each transaction.
    This creates a normal distribution of transactions across users, simulating real-world behavior where some users are more active than others.'''

print("✅ User base synthesized. Total transactions:", len(df))

✅ User base synthesized. Total transactions: 50000


<h3>Phase 2: The RFM Calculation Engine</h3>
<span style="font-size:18px">Now we compress those 50,000 transactions into a mathematical summary of every user's exact value to the company.</span>

In [3]:
# 1. Establish the "Snapshot Date" for Recency calculations
snapshot_date = df['timestamp'].max() + pd.Timedelta(days=1)

'''To calculate 'Recency' (how many days ago someone bought something), I needed a baseline 'today'. 
I programmatically found the absolute last timestamp in the dataset and added exactly one day to it. 
This prevents any user from having a Recency of 0 days, which can cause mathematical errors like 'division by zero' in more complex clustering models.'''

# 2. Perform the RFM Aggregation
rfm = df.groupby('User_ID').agg({
    'timestamp': lambda x: (snapshot_date - x.max()).days,  # Recency
    'User_ID': 'count',                                     # Frequency
    'amount (INR)': 'sum'                                   # Monetary
})

'''This is the core ETL aggregation step. Instead of looping through users, which is computationally expensive and slow, I used Pandas vectorized .agg() function. 
It applies a custom lambda function to find the maximum timestamp for Recency, counts the primary keys for Frequency, and sums the transaction column for the Monetary value.
It's built to scale even if the dataset was 10 million rows.'''

# 3. Clean up the columns for professional presentation
rfm.rename(columns={
    'timestamp': 'Recency',
    'User_ID': 'Frequency',
    'amount (INR)': 'Monetary'
}, inplace=True)

# 4. Save the baseline RFM metrics
rfm.to_csv('rfm_baseline.csv')

print("\n✅ RFM Calculated. Previewing the top 5 users:")
print(rfm.head())


✅ RFM Calculated. Previewing the top 5 users:
           Recency  Frequency  Monetary
User_ID                                
User_0001       22         30     47648
User_0002        1         15     22125
User_0003       58         17     28086
User_0004       23         23     36760
User_0005       71         19     31880


<h3>Phase 3: The Unsupervised Learning Model.</h3>
<span style="font-size:18px">We are going to use K-Means to draw those boundaries. However, before we run the AI, there is one massive mathematical trap we must avoid: <b>Data Scaling</b>.
</span>

In [5]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings('ignore')

# 1. Load the RFM Report Cards
rfm = pd.read_csv('rfm_baseline.csv', index_col='User_ID')

# 2. Scale the Data (The Most Critical Step)
# This forces Recency (e.g., 5 days) and Monetary (e.g., ₹45,000) onto the same playing field
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm)

'''This is the single most important line of code in the model. K-Means is a distance-based algorithm. 
If I don't scale the data, the AI will completely ignore 'Recency' because the numbers are small (like 5 or 12 days), and it will be completely 
overpowered by 'Monetary' because the numbers are massive (like ₹80,000). StandardScaler squashes all three columns down to a level playing field so the AI respects them equally'''

# 3. Train the AI (K-Means Clustering)
# We are asking the AI to find 4 distinct financial personas
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)

'''I chose 4 clusters because, in a real business environment, marketing teams need actionable personas. 
If I told the AI to find 15 clusters, the marketing team wouldn't be able to design 15 different ad campaigns. 
Four gives us a clean split: High-Value Loyalists, At-Risk Spenders, Low-Value Newbies, and Dormant Churned users.'''

# 4. Profile the Clusters to understand the "Personas"
# We calculate the average R, F, and M for each group to see who they are
cluster_summary = rfm.groupby('Cluster').agg({
    'Recency': 'mean',
    'Frequency': 'mean',
    'Monetary': ['mean', 'count']
}).round(1)

'''After the AI assigned a cluster number (0, 1, 2, 3) to every user, I needed to reverse-engineer why the AI grouped them that way. 
By calculating the average Recency, Frequency, and Spend for each cluster, the mathematical personas instantly revealed themselves.'''

# Clean up column names for the printout
cluster_summary.columns = ['Avg_Recency_Days', 'Avg_Frequency', 'Avg_Spend_INR', 'Total_Users']

print("--- AI Clustering Complete ---")
print(cluster_summary)

# 5. Save the final assigned personas
rfm.to_csv('rfm_clusters_final.csv')

--- AI Clustering Complete ---
         Avg_Recency_Days  Avg_Frequency  Avg_Spend_INR  Total_Users
Cluster                                                             
0                    43.9           31.9        53043.7          390
1                   209.9           22.2        33398.9          231
2                    50.8           20.0        27967.6          544
3                    43.7           25.8        39982.6          835


<h3 style="text-align:center"> <b>Report:</b> Overview of the Clusters formed </h3>
<p data-path-to-node="1">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;If I am a DS and presenting this to the Chief Marketing Officer, I wouldn't say "Cluster 0" or "Cluster 1." instead give them identities. Let's translate these numbers into <strong data-path-to-node="1" data-index-in-node="174">Financial Personas</strong>:</p>
<p>-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------</p>
<h3 data-path-to-node="2">👑 Cluster 0: "The Elite Whales" (390 Users)</h3>
<ul data-path-to-node="3">
<li>
<p data-path-to-node="3,0,0"><strong data-path-to-node="3,0,0" data-index-in-node="0">The Math:</strong> They have the highest Frequency (31.9) and by far the highest Monetary spend (₹53,043). Their Recency is extremely low (43.9 days), meaning they are highly active <em data-path-to-node="3,0,0" data-index-in-node="173">right now</em>.</p>
</li>
<li>
<p data-path-to-node="3,1,0"><strong data-path-to-node="3,1,0" data-index-in-node="0">Business Action:</strong> Do not offer them cashback (they already spend money anyway). Offer them VIP perks, airport lounge access, or an exclusive metal credit card to lock in their loyalty.</p>
</li>
</ul>
<h3 data-path-to-node="4">🛑 Cluster 1: "Dormant / Churned" (231 Users)</h3>
<ul data-path-to-node="5">
<li>
<p data-path-to-node="5,0,0"><strong data-path-to-node="5,0,0" data-index-in-node="0">The Math:</strong> Look at that Recency! <strong data-path-to-node="5,0,0" data-index-in-node="32">209.9 days</strong>. These users haven't touched your app in over 7 months. They used to be decent spenders (₹33,398), but they have clearly moved to a competitor app (like PhonePe or GPay).</p>
</li>
<li>
<p data-path-to-node="5,1,0"><strong data-path-to-node="5,1,0" data-index-in-node="0">Business Action:</strong> Aggressive win-back campaigns. Send them a push notification: <em data-path-to-node="5,1,0" data-index-in-node="79">"We miss you! Here is a flat ₹500 cashback on your next UPI transfer."</em></p>
</li>
</ul>
<h3 data-path-to-node="6">🚶 Cluster 2: "Casual Users" (544 Users)</h3>
<ul data-path-to-node="7">
<li>
<p data-path-to-node="7,0,0"><strong data-path-to-node="7,0,0" data-index-in-node="0">The Math:</strong> They are somewhat active (50 days Recency), but they have the absolute lowest Frequency (20.0) and the lowest Monetary spend (₹27,967). They use your app, but only occasionally for small things.</p>
</li>
<li>
<p data-path-to-node="7,1,0"><strong data-path-to-node="7,1,0" data-index-in-node="0">Business Action:</strong> Gamification. Create a "Streak" challenge: <em data-path-to-node="7,1,0" data-index-in-node="60">"Make 5 transactions this week to unlock a mystery reward"</em> to force their Frequency up.</p>
</li>
</ul>
<h3 data-path-to-node="8">🏢 Cluster 3: "Core Mainstream" (835 Users)</h3>
<ul data-path-to-node="9">
<li>
<p data-path-to-node="9,0,0"><strong data-path-to-node="9,0,0" data-index-in-node="0">The Math:</strong> This is your bread and butter. They are your largest group. They are highly active (43.7 days Recency), and sit perfectly in the middle for both spend (₹39,982) and frequency (25.8).</p>
</li>
<li>
<p data-path-to-node="9,1,0"><strong data-path-to-node="9,1,0" data-index-in-node="0">Business Action:</strong> Cross-selling. They trust your app, so it's time to sell them Insurance, Mutual Funds, or fixed deposits.</p>
</li>
</ul>
<p>--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------</p>

<h3>Phase 4: Mapping the Personas to our Data</h3>
<span style="font-size:18px">Now we need to attach these Cluster names to our DataFrame so we can use them in the Streamlit Dashboard.</span>

In [6]:
# 1. Create a mapping dictionary based on our AI output
persona_mapping = {
    0: 'Elite Whales',
    1: 'Dormant / Churned',
    2: 'Casual Users',
    3: 'Core Mainstream'
}

# 2. Apply the names to a new column
rfm['Persona'] = rfm['Cluster'].map(persona_mapping)

# 3. Save the final business-ready dataset!
rfm.to_csv('rfm_clusters_final.csv')
rfm.to_pickle('user_personas.pkl') # Save as pickle for the dashboard

print("✅ Personas mapped and saved successfully!")
print(rfm[['Recency', 'Frequency', 'Monetary', 'Persona']].head())

✅ Personas mapped and saved successfully!
           Recency  Frequency  Monetary          Persona
User_ID                                                 
User_0001       22         30     47648     Elite Whales
User_0002        1         15     22125     Casual Users
User_0003       58         17     28086     Casual Users
User_0004       23         23     36760  Core Mainstream
User_0005       71         19     31880     Casual Users


<h3>Phase 5: Visualizing the AI Clusters</h3>

In [7]:
import plotly.express as px
import plotly.graph_objects as go

print("--- Generating 3D Cluster Visualization ---")

# 1. Create the 3D Scatter Plot
# We map Recency to X, Frequency to Y, and Monetary to Z.
# We color the dots based on the Persona we just created.
fig_3d = px.scatter_3d(
    rfm, 
    x='Recency', 
    y='Frequency', 
    z='Monetary',
    color='Persona',
    title='3D Customer Segmentation (RFM)',
    opacity=0.7,
    size_max=5,
    color_discrete_sequence=px.colors.qualitative.Bold
)

# Tweak the layout to make it look professional
fig_3d.update_layout(margin=dict(l=0, r=0, b=0, t=40))

# Show the interactive chart right here in the notebook!
fig_3d.show()

# 2. Save the figure as an HTML file so we can view it standalone
fig_3d.write_html("rfm_3d_clusters.html")
print("✅ 3D Plot saved as 'rfm_3d_clusters.html'")

# 3. Create a Radar Chart (Spider Web Chart) for the Averages
print("\n--- Generating Persona Radar Chart ---")

# We use the 'cluster_summary' we generated earlier to plot the averages
categories = ['Recency', 'Frequency', 'Monetary']

fig_radar = go.Figure()

for i, row in cluster_summary.iterrows():
    # We need to scale the values down just for the Radar chart so they fit on one screen
    # (Otherwise Monetary at 50,000 will make Recency at 43 invisible)
    # This is a quick normalization trick:
    r_norm = row['Avg_Recency_Days'] / cluster_summary['Avg_Recency_Days'].max()
    f_norm = row['Avg_Frequency'] / cluster_summary['Avg_Frequency'].max()
    m_norm = row['Avg_Spend_INR'] / cluster_summary['Avg_Spend_INR'].max()
    
    # We pull the Persona name from our mapping
    persona_name = persona_mapping[i]

    fig_radar.add_trace(go.Scatterpolar(
        r=[r_norm, f_norm, m_norm],
        theta=categories,
        fill='toself',
        name=persona_name
    ))

fig_radar.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    showlegend=True,
    title="Persona Strengths (Normalized)"
)

fig_radar.show()

--- Generating 3D Cluster Visualization ---


✅ 3D Plot saved as 'rfm_3d_clusters.html'

--- Generating Persona Radar Chart ---


<h4 style="text-align:center"><b>Visualisation logic:</b></h4>
<ul>
<li>
<p data-path-to-node="7,0,0"><strong data-path-to-node="7,0,0" data-index-in-node="0">The 3D Scatter Plot:</strong></p>
<ul data-path-to-node="7,0,1">
<li>
<p data-path-to-node="7,0,1,0,0"><strong data-path-to-node="7,0,1,0,0" data-index-in-node="0">Justification:</strong> "I chose a 3D Scatter Plot because K-Means literally measures the distance between data points in multi-dimensional space. By plotting Recency, Frequency, and Monetary on the X, Y, and Z axes, the stakeholders can visually confirm that the AI accurately drew boundaries around similar users. You can physically spin the graph and see the 'Whales' hovering high up in the Monetary axis."</p>
</li>
</ul>
</li>
<li>
<p data-path-to-node="7,1,0"><strong data-path-to-node="7,1,0" data-index-in-node="0">The Radar Chart (Spider Chart):</strong></p>
<ul data-path-to-node="7,1,1">
<li>
<p data-path-to-node="7,1,1,0,0"><strong data-path-to-node="7,1,1,0,0" data-index-in-node="0">Justification:</strong> "While the 3D plot is great for data scientists, Marketing teams usually prefer Radar charts. It clearly shows the 'shape' of a persona. However, because Recency is in days and Monetary is in thousands of Rupees, I had to apply <strong data-path-to-node="7,1,1,0,0" data-index-in-node="242">Min-Max Normalization</strong> just for the chart so all axes scale from 0.0 to 1.0. This allows us to compare a persona's relative strengths perfectly."</p>
</li>
</ul>
</li>
</ul>